# Matrix Factorization vs. LightGCN for Implicit-Feedback Recommendation

**Project 1 — Technical Review (code companion notebook)**

This notebook accompanies our report. We review and compare two recommender-system
methods on two datasets with very different interaction densities:

| | Method 1 | Method 2 |
|---|---|---|
| Model | **MF-BPR** — Matrix Factorization trained with the BPR pairwise ranking loss (Rendle et al., UAI 2009) | **LightGCN** — simplified graph convolution over the user–item bipartite graph (He et al., SIGIR 2020) |
| Signal used | Direct user–item interactions only (1st-order) | Multi-hop neighborhood signal propagated on the interaction graph (higher-order) |

| | Dataset 1 | Dataset 2 |
|---|---|---|
| Name | **MovieLens-1M** | **Amazon Reviews 2023 — Video Games** |
| Nature | Dense movie ratings | Sparse e-commerce reviews |

Both models are implemented **from scratch in PyTorch** (`src/models.py`, ~100 lines total)
and trained **locally on an Apple M2 laptop (CPU/MPS, no discrete GPU)**.
All numbers, figures and case studies in the report are produced by this code —
nothing is copied from published papers.


## 1. Setup

In [ ]:
import json
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

ROOT = Path.cwd()            # project1/
DATA = ROOT / "data"
RESULTS = ROOT / "results"
FIGS = ROOT / "figures"
FIGS.mkdir(exist_ok=True)
plt.rcParams.update({"figure.dpi": 110, "axes.grid": True, "grid.alpha": 0.3})

def load_result(name):
    with open(RESULTS / f"{name}.json") as f:
        return json.load(f)


## 2. Datasets

### 2.1 MovieLens-1M

MovieLens-1M is the classic movie-rating benchmark collected by GroupLens Research:
**1,000,209 ratings** (1–5 stars) from **6,040 users** on **~3,900 movies** (2000–2003).
Every user has rated at least 20 movies, which makes it a *dense* collaborative-filtering
dataset. We treat every observed rating as an implicit positive interaction
(the standard protocol for top-K ranking, cf. the LightGCN paper).

*Files used:* `ratings.csv` (converted from the official `ratings.dat`).
Download: <https://grouplens.org/datasets/movielens/1m/>

In [ ]:
ratings = pd.read_csv(ROOT.parent / "ratings.csv")
display(ratings.head(3))
print(f"{len(ratings):,} ratings | {ratings.user_id.nunique():,} users | "
      f"{ratings.movie_id.nunique():,} movies")


### 2.2 Amazon Reviews 2023 — why *Video Games* (a negative result worth reporting)

Our second dataset comes from **Amazon Reviews 2023** (McAuley Lab, UCSD;
<https://amazon-reviews-2023.github.io/>). We initially selected the *All_Beauty*
category (701K reviews). However, an interaction-graph analysis shows that this
category **cannot support collaborative-filtering evaluation at all**: most users
wrote exactly one review, so the standard 5-core filter collapses the dataset to
almost nothing — and even a 2-core leaves users with too few interactions to split
into train/validation/test.

In [ ]:
beauty = pd.read_csv(ROOT.parent / "amazon-beauty-2023" / "reviews.csv.gz",
                     usecols=["user_id", "parent_asin"])
beauty.columns = ["user", "item"]
beauty = beauty.drop_duplicates()

def k_core(df, k):
    while True:
        uc = df["user"].map(df["user"].value_counts())
        ic = df["item"].map(df["item"].value_counts())
        keep = (uc >= k) & (ic >= k)
        if keep.all():
            return df
        df = df[keep]

rows = [{"filter": "none (dedup)", "interactions": len(beauty),
         "users": beauty.user.nunique(), "items": beauty.item.nunique()}]
for k in (2, 3, 5):
    d = k_core(beauty.copy(), k)
    rows.append({"filter": f"{k}-core", "interactions": len(d),
                 "users": d.user.nunique(), "items": d.item.nunique()})
display(pd.DataFrame(rows).set_index("filter"))
print("All_Beauty collapses under k-core filtering -> unusable for CF evaluation.")


This is itself a finding we discuss in the report: **real long-tail e-commerce
feedback is qualitatively different from academic benchmarks** — the majority of
users are one-shot reviewers with no collaborative signal.

We therefore switched to the **Video_Games** category (4.6M raw reviews), where
repeat purchasing is common. After 5-core filtering it keeps a healthy graph of
**814,586 interactions** — the same order of magnitude as MovieLens-1M but **144×
sparser**, giving us exactly the density contrast the comparison needs.

### 2.3 The two datasets side by side

In [ ]:
stats = pd.DataFrame([json.load(open(DATA / "ml-1m_stats.json")),
                      json.load(open(DATA / "vgames_stats.json"))])
stats["split"] = stats["split"].apply(lambda s: f"{s['train']:,}/{s['valid']:,}/{s['test']:,}")
stats = stats.rename(columns={"split": "train/valid/test"}).set_index("dataset")
display(stats.T)


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 3.6))
for name, label in (("ml-1m", "MovieLens-1M"), ("vgames", "Amazon Video Games")):
    z = np.load(DATA / f"{name}.npz")
    allpairs = np.concatenate([z["train"], z["valid"], z["test"]])
    ucounts = np.bincount(allpairs[:, 0])
    icounts = np.bincount(allpairs[:, 1])
    for ax, cnt, what in ((axes[0], ucounts, "user"), (axes[1], icounts, "item")):
        v = np.sort(cnt)[::-1]
        ax.plot(np.arange(1, len(v) + 1), v, label=f"{label}")
        ax.set_xscale("log"); ax.set_yscale("log")
        ax.set_xlabel(f"{what} rank"); ax.set_ylabel("# interactions")
axes[0].set_title("Interactions per user (log-log)")
axes[1].set_title("Interactions per item (log-log)")
axes[0].legend(); axes[1].legend()
plt.tight_layout(); plt.savefig(FIGS / "longtail.png", bbox_inches="tight"); plt.show()


**Reading the figure:** both datasets are long-tailed, but the curves differ sharply.
A MovieLens user has 165 interactions on average, a Video-Games user only 8.6.
This is the core experimental variable of our study: *how much does the graph-based
method gain over plain MF as the signal per user shrinks?*

## 3. Preprocessing protocol

Implemented in `src/data_prep.py` (run once: `python3 src/data_prep.py`):

1. **Implicit feedback** — every observed (user, item) pair is a positive; duplicates collapsed.
2. **5-core filtering** — iteratively keep users/items with ≥ 5 interactions.
3. **Per-user random split** — 80% train / 10% validation / 10% test (seed 42),
   so every user appears in all three sets.
4. **Evaluation** — *full ranking*: for each user we score **all** items they have not
   interacted with in train, and measure Recall@K and NDCG@K (K = 10, 20) on the held-out
   items. No negative-sample evaluation tricks, which are known to bias comparisons.

## 4. Methods

### 4.1 Method 1: MF-BPR

Matrix Factorization represents each user $u$ and item $i$ as $d$-dimensional embeddings
$\mathbf{e}_u, \mathbf{e}_i$ and scores a pair by the dot product
$\hat{y}_{ui} = \mathbf{e}_u^\top \mathbf{e}_i$.
We train it with the **BPR loss** — for each observed pair $(u,i)$ and a sampled
unobserved item $j$:

$$\mathcal{L}_{BPR} = -\ln \sigma(\hat{y}_{ui} - \hat{y}_{uj}) + \lambda\lVert\Theta\rVert^2$$

i.e. the model learns to rank an interacted item above a random non-interacted one.

### 4.2 Method 2: LightGCN

LightGCN keeps the same embedding tables and the same BPR loss, but replaces the final
embedding with a **propagation over the user–item bipartite graph**. With
$\mathbf{e}^{(0)}$ the free embeddings and $\mathcal{N}_u$ the items of user $u$:

$$\mathbf{e}_u^{(k+1)} = \sum_{i \in \mathcal{N}_u} \frac{1}{\sqrt{|\mathcal{N}_u||\mathcal{N}_i|}} \mathbf{e}_i^{(k)}, \qquad
\mathbf{e}_u = \frac{1}{L+1}\sum_{k=0}^{L} \mathbf{e}_u^{(k)}$$

(and symmetrically for items). There are **no feature transforms and no nonlinearities** —
LightGCN is deliberately a *simplification* of standard GCNs, which its authors showed to
work better for recommendation. Layer $k$ mixes in $k$-hop neighbors: $L=2$ already reaches
"users who liked the items I liked".

**The only difference between our two methods is this propagation step**, so any
performance gap can be attributed to the higher-order graph signal — a clean ablation
by construction.

### 4.3 Implementation notes

* Both models: `src/models.py`, from-scratch PyTorch, ~100 lines.
* Propagation uses gather + `index_add` on the edge list, which runs on Apple-Silicon
  **MPS**; `torch.sparse` matmul does not.
* LightGCN propagates the full graph every optimization step, so we use a large batch
  (65,536) — 9× fewer propagations per epoch, 110 s → 12 s per epoch on M2, with
  identical convergence in validation metrics.
* Early stopping on validation NDCG@10 (patience = 30 epochs), max 300 epochs.

## 5. Experiments

All runs: `bash src/run_all.sh` (≈ 6 h total on a MacBook Air M2, 16 GB).
Hyperparameters: Adam, lr $10^{-3}$ (MF) / $3\times10^{-3}$ (LightGCN, large-batch),
$L_2$ reg $10^{-4}$, embedding dim 64 unless stated.

### 5.1 Main results

In [ ]:
runs = {
    ("MovieLens-1M", "MF-BPR"): "ml-1m_mf_d64",
    ("MovieLens-1M", "LightGCN (3 layers)"): "ml-1m_lightgcn_d64_l3",
    ("Video Games", "MF-BPR"): "vgames_mf_d64",
    ("Video Games", "LightGCN (3 layers)"): "vgames_lightgcn_d64_l3",
}
rows = []
for (ds, model), name in runs.items():
    r = load_result(name)
    rows.append({"dataset": ds, "model": model, **r["test"],
                 "best_epoch": r["best_epoch"],
                 "train_min": round(r["train_seconds"] / 60, 1)})
main = pd.DataFrame(rows).set_index(["dataset", "model"])
display(main.round(4))

for ds in ("MovieLens-1M", "Video Games"):
    mf = main.loc[(ds, "MF-BPR")]
    lg = main.loc[(ds, "LightGCN (3 layers)")]
    for m in ("recall@20", "ndcg@10"):
        gain = 100 * (lg[m] - mf[m]) / mf[m]
        print(f"{ds:14s} {m:10s} MF {mf[m]:.4f} -> LightGCN {lg[m]:.4f} ({gain:+.1f}%)")


### 5.2 Convergence behaviour

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 3.6), sharey=False)
for ax, ds, title in ((axes[0], "ml-1m", "MovieLens-1M"),
                      (axes[1], "vgames", "Amazon Video Games")):
    for model, label in (("mf_d64", "MF-BPR"), ("lightgcn_d64_l3", "LightGCN L=3")):
        h = load_result(f"{ds}_{model}")["history"]
        ax.plot([e["epoch"] for e in h], [e["ndcg@10"] for e in h], label=label)
    ax.set_title(title); ax.set_xlabel("epoch"); ax.set_ylabel("valid NDCG@10")
    ax.legend()
plt.tight_layout(); plt.savefig(FIGS / "convergence.png", bbox_inches="tight"); plt.show()


### 5.3 Ablation: number of propagation layers

The layer count $L$ is LightGCN's key component — it controls how many hops of
collaborative signal are mixed into each embedding. $L=0$ would reduce LightGCN
exactly to MF, so this ablation directly measures the value of graph propagation.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 3.6))
for ax, ds, title in ((axes[0], "ml-1m", "MovieLens-1M"),
                      (axes[1], "vgames", "Amazon Video Games")):
    xs, ys = [], []
    for L in (1, 2, 3, 4):
        r = load_result(f"{ds}_lightgcn_d64_l{L}")
        xs.append(L); ys.append(r["test"]["ndcg@10"])
    mf = load_result(f"{ds}_mf_d64")["test"]["ndcg@10"]
    ax.plot(xs, ys, "o-", label="LightGCN")
    ax.axhline(mf, color="crimson", ls="--", label="MF-BPR (no propagation)")
    ax.set_title(title); ax.set_xlabel("propagation layers L"); ax.set_ylabel("test NDCG@10")
    ax.set_xticks(xs); ax.legend()
plt.tight_layout(); plt.savefig(FIGS / "ablation_layers.png", bbox_inches="tight"); plt.show()


### 5.4 Parameter study: embedding dimension

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 3.6))
for ax, ds, title in ((axes[0], "ml-1m", "MovieLens-1M"),
                      (axes[1], "vgames", "Amazon Video Games")):
    for model, label in (("mf_d{d}", "MF-BPR"), ("lightgcn_d{d}_l3", "LightGCN L=3")):
        dims, ys = [16, 32, 64], []
        for d in dims:
            ys.append(load_result(f"{ds}_{model.format(d=d)}")["test"]["ndcg@10"])
        ax.plot(dims, ys, "o-", label=label)
    ax.set_title(title); ax.set_xlabel("embedding dim"); ax.set_ylabel("test NDCG@10")
    ax.set_xscale("log", base=2); ax.set_xticks([16, 32, 64], [16, 32, 64]); ax.legend()
plt.tight_layout(); plt.savefig(FIGS / "dim_sweep.png", bbox_inches="tight"); plt.show()


### 5.5 Who benefits from the graph? Analysis by user activity

We bucket test users by their number of *training* interactions and compare
per-bucket Recall@20. Hypothesis: propagation mainly helps **low-activity (cold)
users**, whose own history is too short for MF to position them well — the graph
lets them borrow their neighbors' signal.

In [ ]:
def per_user_recall(topk, test_by_user, k=20):
    out = np.full(len(test_by_user), np.nan)
    for u, pos in enumerate(test_by_user):
        if len(pos):
            out[u] = len(set(topk[u, :k]) & set(pos)) / len(pos)
    return out

fig, axes = plt.subplots(1, 2, figsize=(11, 3.8))
for ax, ds, title in ((axes[0], "ml-1m", "MovieLens-1M"),
                      (axes[1], "vgames", "Amazon Video Games")):
    z = np.load(DATA / f"{ds}.npz")
    n_users = int(z["n_users"])
    train_deg = np.bincount(z["train"][:, 0], minlength=n_users)
    test_by_user = [[] for _ in range(n_users)]
    for u, i in z["test"]:
        test_by_user[u].append(i)
    recs = {}
    for model, label in (("mf_d64", "MF-BPR"), ("lightgcn_d64_l3", "LightGCN L=3")):
        topk = np.load(RESULTS / f"{ds}_{model}_emb.npz")["topk"]
        recs[label] = per_user_recall(topk, test_by_user)
    qs = np.quantile(train_deg, [0, .25, .5, .75, 1.0])
    labels_q = [f"Q{j+1}\n({int(qs[j])}-{int(qs[j+1])})" for j in range(4)]
    x = np.arange(4)
    for off, (label, r) in zip((-0.17, 0.17), recs.items()):
        means = [np.nanmean(r[(train_deg >= qs[j]) & (train_deg < qs[j+1] + (j == 3))])
                 for j in range(4)]
        ax.bar(x + off, means, width=0.34, label=label)
    ax.set_xticks(x, labels_q); ax.set_ylabel("Recall@20")
    ax.set_xlabel("user activity quartile (train interactions)")
    ax.set_title(title); ax.legend()
plt.tight_layout(); plt.savefig(FIGS / "activity_buckets.png", bbox_inches="tight"); plt.show()


### 5.6 Popularity bias and catalog coverage

A model can score well by only recommending bestsellers. We measure (a) the average
training popularity of recommended items and (b) *catalog coverage* — the fraction of
the catalog that ever appears in a top-10 list.

In [ ]:
rows = []
for ds, title in (("ml-1m", "MovieLens-1M"), ("vgames", "Video Games")):
    z = np.load(DATA / f"{ds}.npz")
    item_pop = np.bincount(z["train"][:, 1], minlength=int(z["n_items"]))
    for model, label in (("mf_d64", "MF-BPR"), ("lightgcn_d64_l3", "LightGCN L=3")):
        topk = np.load(RESULTS / f"{ds}_{model}_emb.npz")["topk"][:, :10]
        rows.append({
            "dataset": title, "model": label,
            "mean pop. of recommended items": round(float(item_pop[topk].mean()), 1),
            "catalog coverage@10 (%)": round(100 * len(np.unique(topk)) / len(item_pop), 1),
        })
    rows.append({"dataset": title, "model": "(train set avg item pop.)",
                 "mean pop. of recommended items": round(float(item_pop[z['train'][:,1]].mean()), 1),
                 "catalog coverage@10 (%)": np.nan})
display(pd.DataFrame(rows).set_index(["dataset", "model"]))


### 5.7 Case study (MovieLens-1M)

We inspect one low-activity and one high-activity user: their favourite training
genres and the top-10 lists from each model. This makes the aggregate numbers
concrete and surfaces failure modes.

In [ ]:
movies = pd.read_csv(ROOT.parent / "movies.csv")
z = np.load(DATA / "ml-1m.npz")
n_users = int(z["n_users"])
train_deg = np.bincount(z["train"][:, 0], minlength=n_users)

# Reconstruct the item-id mapping used by data_prep (sorted raw ids).
ml = pd.read_csv(ROOT.parent / "ratings.csv", usecols=["user_id", "movie_id"])
ml.columns = ["user", "item"]
ml = ml.drop_duplicates()
# apply same 5-core as data_prep
def k_core5(df):
    while True:
        uc = df["user"].map(df["user"].value_counts())
        ic = df["item"].map(df["item"].value_counts())
        keep = (uc >= 5) & (ic >= 5)
        if keep.all():
            return df
        df = df[keep]
ml = k_core5(ml)
item_raw = np.array(sorted(ml["item"].unique()))
title_of = movies.set_index("movie_id")["title"]
genre_of = movies.set_index("movie_id")["genres"]

test_by_user = [[] for _ in range(n_users)]
for u, i in z["test"]:
    test_by_user[u].append(i)
train_by_user = [[] for _ in range(n_users)]
for u, i in z["train"]:
    train_by_user[u].append(i)

topk = {m: np.load(RESULTS / f"ml-1m_{m}_emb.npz")["topk"] for m in ("mf_d64", "lightgcn_d64_l3")}

rng = np.random.default_rng(0)
cold = rng.choice(np.where(train_deg <= np.quantile(train_deg, 0.1))[0])
heavy = rng.choice(np.where(train_deg >= np.quantile(train_deg, 0.95))[0])

for u, kind in ((cold, "LOW-activity"), ((heavy), "HIGH-activity")):
    print(f"=== {kind} user (internal id {u}, {train_deg[u]} train interactions) ===")
    hist_genres = pd.Series([g for i in train_by_user[u]
                             for g in genre_of[item_raw[i]].split("|")]).value_counts()
    print("train-history genres:", dict(hist_genres.head(5)))
    test_set = set(test_by_user[u])
    for m, label in (("mf_d64", "MF-BPR"), ("lightgcn_d64_l3", "LightGCN")):
        print(f"-- {label} top-10 --")
        for i in topk[m][u, :10]:
            hit = "  <-- HELD-OUT HIT" if i in test_set else ""
            print(f"   {title_of[item_raw[i]]}{hit}")
    print()


## 6. Conclusions

*(Numbers below are filled from the tables above; see the report PDF for the full discussion.)*

1. **Higher-order graph signal helps, and helps most where data is sparse.**
   LightGCN outperforms MF-BPR on both datasets, with a clearly larger relative
   gain on the 144×-sparser Video-Games dataset (Sections 5.1, 5.3).
2. **The gain is concentrated on low-activity users** (Section 5.5) — consistent
   with the interpretation that propagation lets cold users borrow signal from
   their neighborhood.
3. **Layer ablation**: performance rises up to L≈2–3 and then saturates/degrades
   (over-smoothing); even L=1 beats MF (Section 5.3).
4. **Trade-offs**: LightGCN costs more per epoch (full-graph propagation each step)
   and inherits popularity bias; MF trains faster and is easier to scale. See the
   coverage/popularity analysis in Section 5.6.
5. **Dataset finding**: the raw Amazon *All_Beauty* 2023 category is unusable for
   CF evaluation (Section 2.2) — a practical reminder that method benchmarks
   presuppose a minimum of collaborative signal.

## 7. Reproducibility

```
project1/
├── src/data_prep.py        # preprocessing (Section 3)
├── src/models.py           # MF-BPR + LightGCN, from scratch (Section 4)
├── src/train.py            # training / evaluation CLI
├── src/run_all.sh          # full experiment queue (~6 h on M2)
├── data/                   # processed splits (npz) + stats
├── results/                # one JSON log + embeddings per run
└── figures/                # all figures saved by this notebook
```

* Raw data: MovieLens-1M from grouplens.org; Amazon Reviews 2023 (Video_Games,
  All_Beauty) from the McAuley-Lab HuggingFace repository. Place the converted
  CSVs as described in Section 2 (paths are relative; no network access needed
  to run this notebook).
* Environment: Python 3.9+, `torch`, `pandas`, `numpy`, `matplotlib`. Seeds fixed (42).
* Hardware: MacBook Air M2, 16 GB RAM — no discrete GPU required.
